# LayerPassLog Anatomy

Audits per-pass layer records, including tensor, graph, module, timing, and gradient metadata.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.log_forward_pass(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].activation.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.passes.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.passes.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.log_forward_pass(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.log_forward_pass(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "ModelLog": log,
    "LayerLog": layer_log,
    "LayerPassLog": layer_pass,
    "ModuleLog": module_log,
    "ModulePassLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnPassLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer passes, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## LayerPassLog Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerPassLog",
    "tl.LayerPassLog.DEFAULT_FILL_STATE",
    "tl.LayerPassLog.FORK_POLICY",
    "tl.LayerPassLog.PORTABLE_STATE_SPEC",
    "tl.LayerPassLog._construction_done",
    "tl.LayerPassLog._pass_finished",
    "tl.LayerPassLog.activation",
    "tl.LayerPassLog.activation_postfunc",
    "tl.LayerPassLog.activation_transform",
    "tl.LayerPassLog.args_captured",
    "tl.LayerPassLog.autograd_saved_bytes",
    "tl.LayerPassLog.autograd_saved_tensor_count",
    "tl.LayerPassLog.bool_conditional_id",
    "tl.LayerPassLog.bool_context_kind",
    "tl.LayerPassLog.bool_is_branch",
    "tl.LayerPassLog.bool_wrapper_kind",
    "tl.LayerPassLog.buffer_address",
    "tl.LayerPassLog.buffer_parent",
    "tl.LayerPassLog.buffer_pass",
    "tl.LayerPassLog.bytes_delta_at_call",
    "tl.LayerPassLog.bytes_peak_at_call",
    "tl.LayerPassLog.captured_arg_template",
    "tl.LayerPassLog.captured_args",
    "tl.LayerPassLog.captured_kwarg_template",
    "tl.LayerPassLog.captured_kwargs",
    "tl.LayerPassLog.child_layers",
    "tl.LayerPassLog.children_tensor_versions",
    "tl.LayerPassLog.co_parent_layers",
    "tl.LayerPassLog.cond_branch_children_by_cond",
    "tl.LayerPassLog.cond_branch_elif_children",
    "tl.LayerPassLog.cond_branch_else_children",
    "tl.LayerPassLog.cond_branch_start_children",
    "tl.LayerPassLog.cond_branch_then_children",
    "tl.LayerPassLog.conditional_branch_depth",
    "tl.LayerPassLog.conditional_branch_stack",
    "tl.LayerPassLog.container_spec",
    "tl.LayerPassLog.containing_module",
    "tl.LayerPassLog.containing_modules",
    "tl.LayerPassLog.copy",
    "tl.LayerPassLog.corresponding_grad_fn",
    "tl.LayerPassLog.creation_order",
    "tl.LayerPassLog.detach_saved_tensor",
    "tl.LayerPassLog.edge_uses",
    "tl.LayerPassLog.equivalent_operations",
    "tl.LayerPassLog.extra_data",
    "tl.LayerPassLog.feeds_output",
    "tl.LayerPassLog.flops_backward",
    "tl.LayerPassLog.flops_forward",
    "tl.LayerPassLog.func_applied",
    "tl.LayerPassLog.func_argnames",
    "tl.LayerPassLog.func_autocast_state",
    "tl.LayerPassLog.func_call_id",
    "tl.LayerPassLog.func_call_stack",
    "tl.LayerPassLog.func_config",
    "tl.LayerPassLog.func_is_inplace",
    "tl.LayerPassLog.func_kwargs_non_tensor",
    "tl.LayerPassLog.func_name",
    "tl.LayerPassLog.func_non_tensor_args",
    "tl.LayerPassLog.func_positional_args_non_tensor",
    "tl.LayerPassLog.func_rng_states",
    "tl.LayerPassLog.func_time",
    "tl.LayerPassLog.get_child_layers",
    "tl.LayerPassLog.get_parent_layers",
    "tl.LayerPassLog.grad_dtype",
    "tl.LayerPassLog.grad_fn_id",
    "tl.LayerPassLog.grad_fn_name",
    "tl.LayerPassLog.grad_fn_object",
    "tl.LayerPassLog.grad_memory",
    "tl.LayerPassLog.grad_memory_str",
    "tl.LayerPassLog.grad_shape",
    "tl.LayerPassLog.gradient",
    "tl.LayerPassLog.has_child_tensor_variations",
    "tl.LayerPassLog.has_children",
    "tl.LayerPassLog.has_co_parents",
    "tl.LayerPassLog.has_gradient",
    "tl.LayerPassLog.has_input_ancestor",
    "tl.LayerPassLog.has_internally_initialized_ancestor",
    "tl.LayerPassLog.has_parents",
    "tl.LayerPassLog.has_saved_activations",
    "tl.LayerPassLog.has_siblings",
    "tl.LayerPassLog.in_cond_branch",
    "tl.LayerPassLog.input_ancestors",
    "tl.LayerPassLog.internally_initialized_ancestors",
    "tl.LayerPassLog.internally_initialized_parents",
    "tl.LayerPassLog.intervention_log",
    "tl.LayerPassLog.io_role",
    "tl.LayerPassLog.is_buffer_layer",
    "tl.LayerPassLog.is_computed_inside_submodule",
    "tl.LayerPassLog.is_final_output",
    "tl.LayerPassLog.is_input_layer",
    "tl.LayerPassLog.is_internally_initialized",
    "tl.LayerPassLog.is_internally_terminated",
    "tl.LayerPassLog.is_leaf_module_output",
    "tl.LayerPassLog.is_output_ancestor",
    "tl.LayerPassLog.is_output_layer",
    "tl.LayerPassLog.is_part_of_iterable_output",
    "tl.LayerPassLog.is_scalar_bool",
    "tl.LayerPassLog.is_submodule_input",
    "tl.LayerPassLog.is_submodule_output",
    "tl.LayerPassLog.is_terminal_bool_layer",
    "tl.LayerPassLog.iterable_output_index",
    "tl.LayerPassLog.layer_label",
    "tl.LayerPassLog.layer_label_no_pass",
    "tl.LayerPassLog.layer_label_no_pass_short",
    "tl.LayerPassLog.layer_label_raw",
    "tl.LayerPassLog.layer_label_short",
    "tl.LayerPassLog.layer_label_w_pass",
    "tl.LayerPassLog.layer_label_w_pass_short",
    "tl.LayerPassLog.layer_total_num",
    "tl.LayerPassLog.layer_type",
    "tl.LayerPassLog.layer_type_num",
    "tl.LayerPassLog.leaf_module_pass",
    "tl.LayerPassLog.log_tensor_grad",
    "tl.LayerPassLog.lookup_keys",
    "tl.LayerPassLog.macs_backward",
    "tl.LayerPassLog.macs_forward",
    "tl.LayerPassLog.materialize_activation",
    "tl.LayerPassLog.materialize_gradient",
    "tl.LayerPassLog.max_distance_from_input",
    "tl.LayerPassLog.max_distance_from_output",
    "tl.LayerPassLog.min_distance_from_input",
    "tl.LayerPassLog.min_distance_from_output",
    "tl.LayerPassLog.module_address_normalized",
    "tl.LayerPassLog.module_entry_exit_thread_output",
    "tl.LayerPassLog.module_entry_exit_threads_inputs",
    "tl.LayerPassLog.module_nesting_depth",
    "tl.LayerPassLog.module_passes_entered",
    "tl.LayerPassLog.module_passes_exited",
    "tl.LayerPassLog.modules_entered",
    "tl.LayerPassLog.modules_entered_argnames",
    "tl.LayerPassLog.modules_exited",
    "tl.LayerPassLog.num_args",
    "tl.LayerPassLog.num_keyword_args",
    "tl.LayerPassLog.num_param_tensors",
    "tl.LayerPassLog.num_params_frozen",
    "tl.LayerPassLog.num_params_total",
    "tl.LayerPassLog.num_params_trainable",
    "tl.LayerPassLog.num_passes",
    "tl.LayerPassLog.num_positional_args",
    "tl.LayerPassLog.operation_equivalence_type",
    "tl.LayerPassLog.operation_num",
    "tl.LayerPassLog.output_descendants",
    "tl.LayerPassLog.output_device",
    "tl.LayerPassLog.output_path",
    "tl.LayerPassLog.params",
    "tl.LayerPassLog.params_memory",
    "tl.LayerPassLog.params_memory_str",
    "tl.LayerPassLog.parent_layer_arg_locs",
    "tl.LayerPassLog.parent_layers",
    "tl.LayerPassLog.parent_param_barcodes",
    "tl.LayerPassLog.parent_param_logs",
    "tl.LayerPassLog.parent_param_passes",
    "tl.LayerPassLog.parent_param_shapes",
    "tl.LayerPassLog.parent_params",
    "tl.LayerPassLog.pass_num",
    "tl.LayerPassLog.passes",
    "tl.LayerPassLog.recurrent_group",
    "tl.LayerPassLog.root_ancestors",
    "tl.LayerPassLog.save_gradients",
    "tl.LayerPassLog.save_tensor_data",
    "tl.LayerPassLog.scalar_bool_value",
    "tl.LayerPassLog.show",
    "tl.LayerPassLog.sibling_layers",
    "tl.LayerPassLog.source_model_log",
    "tl.LayerPassLog.tensor",
    "tl.LayerPassLog.tensor_dtype",
    "tl.LayerPassLog.tensor_label_raw",
    "tl.LayerPassLog.tensor_memory",
    "tl.LayerPassLog.tensor_memory_str",
    "tl.LayerPassLog.tensor_shape",
    "tl.LayerPassLog.transformed_activation",
    "tl.LayerPassLog.transformed_activation_dtype",
    "tl.LayerPassLog.transformed_activation_memory",
    "tl.LayerPassLog.transformed_activation_shape",
    "tl.LayerPassLog.transformed_gradient",
    "tl.LayerPassLog.transformed_gradient_dtype",
    "tl.LayerPassLog.transformed_gradient_memory",
    "tl.LayerPassLog.transformed_gradient_shape",
    "tl.LayerPassLog.uses_params",
]

pretty_print_fields(
    layer_pass,
    [
        "layer_label",
        "layer_label_w_pass",
        "pass_num",
        "func_name",
        "tensor_shape",
        "tensor_dtype",
        "has_gradient",
        "flops_forward",
        "macs_forward",
    ],
)
print("copy type", type(layer_pass.copy()).__name__)

## Identity and tensor data

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerPassLog",
    "tl.LayerPassLog.DEFAULT_FILL_STATE",
    "tl.LayerPassLog.FORK_POLICY",
    "tl.LayerPassLog.PORTABLE_STATE_SPEC",
    "tl.LayerPassLog._construction_done",
    "tl.LayerPassLog._pass_finished",
    "tl.LayerPassLog.activation",
    "tl.LayerPassLog.activation_postfunc",
    "tl.LayerPassLog.activation_transform",
    "tl.LayerPassLog.args_captured",
    "tl.LayerPassLog.autograd_saved_bytes",
    "tl.LayerPassLog.autograd_saved_tensor_count",
    "tl.LayerPassLog.bool_conditional_id",
    "tl.LayerPassLog.bool_context_kind",
    "tl.LayerPassLog.bool_is_branch",
    "tl.LayerPassLog.bool_wrapper_kind",
    "tl.LayerPassLog.buffer_address",
    "tl.LayerPassLog.buffer_parent",
    "tl.LayerPassLog.buffer_pass",
    "tl.LayerPassLog.bytes_delta_at_call",
    "tl.LayerPassLog.bytes_peak_at_call",
    "tl.LayerPassLog.captured_arg_template",
    "tl.LayerPassLog.captured_args",
    "tl.LayerPassLog.captured_kwarg_template",
    "tl.LayerPassLog.captured_kwargs",
    "tl.LayerPassLog.child_layers",
]

audit_touch_items("Identity and tensor data", ITEMS, AUDIT_CONTEXT)

## Function metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerPassLog.children_tensor_versions",
    "tl.LayerPassLog.co_parent_layers",
    "tl.LayerPassLog.cond_branch_children_by_cond",
    "tl.LayerPassLog.cond_branch_elif_children",
    "tl.LayerPassLog.cond_branch_else_children",
    "tl.LayerPassLog.cond_branch_start_children",
    "tl.LayerPassLog.cond_branch_then_children",
    "tl.LayerPassLog.conditional_branch_depth",
    "tl.LayerPassLog.conditional_branch_stack",
    "tl.LayerPassLog.container_spec",
    "tl.LayerPassLog.containing_module",
    "tl.LayerPassLog.containing_modules",
    "tl.LayerPassLog.copy",
    "tl.LayerPassLog.corresponding_grad_fn",
    "tl.LayerPassLog.creation_order",
    "tl.LayerPassLog.detach_saved_tensor",
    "tl.LayerPassLog.edge_uses",
    "tl.LayerPassLog.equivalent_operations",
    "tl.LayerPassLog.extra_data",
    "tl.LayerPassLog.feeds_output",
    "tl.LayerPassLog.flops_backward",
    "tl.LayerPassLog.flops_forward",
    "tl.LayerPassLog.func_applied",
    "tl.LayerPassLog.func_argnames",
    "tl.LayerPassLog.func_autocast_state",
    "tl.LayerPassLog.func_call_id",
]

audit_touch_items("Function metadata", ITEMS, AUDIT_CONTEXT)

## Autograd and gradients

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerPassLog.func_call_stack",
    "tl.LayerPassLog.func_config",
    "tl.LayerPassLog.func_is_inplace",
    "tl.LayerPassLog.func_kwargs_non_tensor",
    "tl.LayerPassLog.func_name",
    "tl.LayerPassLog.func_non_tensor_args",
    "tl.LayerPassLog.func_positional_args_non_tensor",
    "tl.LayerPassLog.func_rng_states",
    "tl.LayerPassLog.func_time",
    "tl.LayerPassLog.get_child_layers",
    "tl.LayerPassLog.get_parent_layers",
    "tl.LayerPassLog.grad_dtype",
    "tl.LayerPassLog.grad_fn_id",
    "tl.LayerPassLog.grad_fn_name",
    "tl.LayerPassLog.grad_fn_object",
    "tl.LayerPassLog.grad_memory",
    "tl.LayerPassLog.grad_memory_str",
    "tl.LayerPassLog.grad_shape",
    "tl.LayerPassLog.gradient",
    "tl.LayerPassLog.has_child_tensor_variations",
    "tl.LayerPassLog.has_children",
    "tl.LayerPassLog.has_co_parents",
    "tl.LayerPassLog.has_gradient",
    "tl.LayerPassLog.has_input_ancestor",
    "tl.LayerPassLog.has_internally_initialized_ancestor",
    "tl.LayerPassLog.has_parents",
]

audit_touch_items("Autograd and gradients", ITEMS, AUDIT_CONTEXT)

## Graph relationships

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerPassLog.has_saved_activations",
    "tl.LayerPassLog.has_siblings",
    "tl.LayerPassLog.in_cond_branch",
    "tl.LayerPassLog.input_ancestors",
    "tl.LayerPassLog.internally_initialized_ancestors",
    "tl.LayerPassLog.internally_initialized_parents",
    "tl.LayerPassLog.intervention_log",
    "tl.LayerPassLog.io_role",
    "tl.LayerPassLog.is_buffer_layer",
    "tl.LayerPassLog.is_computed_inside_submodule",
    "tl.LayerPassLog.is_final_output",
    "tl.LayerPassLog.is_input_layer",
    "tl.LayerPassLog.is_internally_initialized",
    "tl.LayerPassLog.is_internally_terminated",
    "tl.LayerPassLog.is_leaf_module_output",
    "tl.LayerPassLog.is_output_ancestor",
    "tl.LayerPassLog.is_output_layer",
    "tl.LayerPassLog.is_part_of_iterable_output",
    "tl.LayerPassLog.is_scalar_bool",
    "tl.LayerPassLog.is_submodule_input",
    "tl.LayerPassLog.is_submodule_output",
    "tl.LayerPassLog.is_terminal_bool_layer",
    "tl.LayerPassLog.iterable_output_index",
    "tl.LayerPassLog.layer_label",
    "tl.LayerPassLog.layer_label_no_pass",
    "tl.LayerPassLog.layer_label_no_pass_short",
]

audit_touch_items("Graph relationships", ITEMS, AUDIT_CONTEXT)

## Module metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerPassLog.layer_label_raw",
    "tl.LayerPassLog.layer_label_short",
    "tl.LayerPassLog.layer_label_w_pass",
    "tl.LayerPassLog.layer_label_w_pass_short",
    "tl.LayerPassLog.layer_total_num",
    "tl.LayerPassLog.layer_type",
    "tl.LayerPassLog.layer_type_num",
    "tl.LayerPassLog.leaf_module_pass",
    "tl.LayerPassLog.log_tensor_grad",
    "tl.LayerPassLog.lookup_keys",
    "tl.LayerPassLog.macs_backward",
    "tl.LayerPassLog.macs_forward",
    "tl.LayerPassLog.materialize_activation",
    "tl.LayerPassLog.materialize_gradient",
    "tl.LayerPassLog.max_distance_from_input",
    "tl.LayerPassLog.max_distance_from_output",
    "tl.LayerPassLog.min_distance_from_input",
    "tl.LayerPassLog.min_distance_from_output",
    "tl.LayerPassLog.module_address_normalized",
    "tl.LayerPassLog.module_entry_exit_thread_output",
    "tl.LayerPassLog.module_entry_exit_threads_inputs",
    "tl.LayerPassLog.module_nesting_depth",
    "tl.LayerPassLog.module_passes_entered",
    "tl.LayerPassLog.module_passes_exited",
    "tl.LayerPassLog.modules_entered",
    "tl.LayerPassLog.modules_entered_argnames",
]

audit_touch_items("Module metadata", ITEMS, AUDIT_CONTEXT)

## Control-flow and buffering

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerPassLog.modules_exited",
    "tl.LayerPassLog.num_args",
    "tl.LayerPassLog.num_keyword_args",
    "tl.LayerPassLog.num_param_tensors",
    "tl.LayerPassLog.num_params_frozen",
    "tl.LayerPassLog.num_params_total",
    "tl.LayerPassLog.num_params_trainable",
    "tl.LayerPassLog.num_passes",
    "tl.LayerPassLog.num_positional_args",
    "tl.LayerPassLog.operation_equivalence_type",
    "tl.LayerPassLog.operation_num",
    "tl.LayerPassLog.output_descendants",
    "tl.LayerPassLog.output_device",
    "tl.LayerPassLog.output_path",
    "tl.LayerPassLog.params",
    "tl.LayerPassLog.params_memory",
    "tl.LayerPassLog.params_memory_str",
    "tl.LayerPassLog.parent_layer_arg_locs",
    "tl.LayerPassLog.parent_layers",
    "tl.LayerPassLog.parent_param_barcodes",
    "tl.LayerPassLog.parent_param_logs",
    "tl.LayerPassLog.parent_param_passes",
    "tl.LayerPassLog.parent_param_shapes",
    "tl.LayerPassLog.parent_params",
    "tl.LayerPassLog.pass_num",
    "tl.LayerPassLog.passes",
]

audit_touch_items("Control-flow and buffering", ITEMS, AUDIT_CONTEXT)

## Remaining pass fields

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerPassLog.recurrent_group",
    "tl.LayerPassLog.root_ancestors",
    "tl.LayerPassLog.save_gradients",
    "tl.LayerPassLog.save_tensor_data",
    "tl.LayerPassLog.scalar_bool_value",
    "tl.LayerPassLog.show",
    "tl.LayerPassLog.sibling_layers",
    "tl.LayerPassLog.source_model_log",
    "tl.LayerPassLog.tensor",
    "tl.LayerPassLog.tensor_dtype",
    "tl.LayerPassLog.tensor_label_raw",
    "tl.LayerPassLog.tensor_memory",
    "tl.LayerPassLog.tensor_memory_str",
    "tl.LayerPassLog.tensor_shape",
    "tl.LayerPassLog.transformed_activation",
    "tl.LayerPassLog.transformed_activation_dtype",
    "tl.LayerPassLog.transformed_activation_memory",
    "tl.LayerPassLog.transformed_activation_shape",
    "tl.LayerPassLog.transformed_gradient",
    "tl.LayerPassLog.transformed_gradient_dtype",
    "tl.LayerPassLog.transformed_gradient_memory",
    "tl.LayerPassLog.transformed_gradient_shape",
    "tl.LayerPassLog.uses_params",
]

audit_touch_items("Remaining pass fields", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerPassLog",
    "tl.LayerPassLog.DEFAULT_FILL_STATE",
    "tl.LayerPassLog.FORK_POLICY",
    "tl.LayerPassLog.PORTABLE_STATE_SPEC",
    "tl.LayerPassLog._construction_done",
    "tl.LayerPassLog._pass_finished",
    "tl.LayerPassLog.activation",
    "tl.LayerPassLog.activation_postfunc",
    "tl.LayerPassLog.activation_transform",
    "tl.LayerPassLog.args_captured",
    "tl.LayerPassLog.autograd_saved_bytes",
    "tl.LayerPassLog.autograd_saved_tensor_count",
    "tl.LayerPassLog.bool_conditional_id",
    "tl.LayerPassLog.bool_context_kind",
    "tl.LayerPassLog.bool_is_branch",
    "tl.LayerPassLog.bool_wrapper_kind",
    "tl.LayerPassLog.buffer_address",
    "tl.LayerPassLog.buffer_parent",
    "tl.LayerPassLog.buffer_pass",
    "tl.LayerPassLog.bytes_delta_at_call",
    "tl.LayerPassLog.bytes_peak_at_call",
    "tl.LayerPassLog.captured_arg_template",
    "tl.LayerPassLog.captured_args",
    "tl.LayerPassLog.captured_kwarg_template",
    "tl.LayerPassLog.captured_kwargs",
    "tl.LayerPassLog.child_layers",
    "tl.LayerPassLog.children_tensor_versions",
    "tl.LayerPassLog.co_parent_layers",
    "tl.LayerPassLog.cond_branch_children_by_cond",
    "tl.LayerPassLog.cond_branch_elif_children",
    "tl.LayerPassLog.cond_branch_else_children",
    "tl.LayerPassLog.cond_branch_start_children",
    "tl.LayerPassLog.cond_branch_then_children",
    "tl.LayerPassLog.conditional_branch_depth",
    "tl.LayerPassLog.conditional_branch_stack",
    "tl.LayerPassLog.container_spec",
    "tl.LayerPassLog.containing_module",
    "tl.LayerPassLog.containing_modules",
    "tl.LayerPassLog.copy",
    "tl.LayerPassLog.corresponding_grad_fn",
    "tl.LayerPassLog.creation_order",
    "tl.LayerPassLog.detach_saved_tensor",
    "tl.LayerPassLog.edge_uses",
    "tl.LayerPassLog.equivalent_operations",
    "tl.LayerPassLog.extra_data",
    "tl.LayerPassLog.feeds_output",
    "tl.LayerPassLog.flops_backward",
    "tl.LayerPassLog.flops_forward",
    "tl.LayerPassLog.func_applied",
    "tl.LayerPassLog.func_argnames",
    "tl.LayerPassLog.func_autocast_state",
    "tl.LayerPassLog.func_call_id",
    "tl.LayerPassLog.func_call_stack",
    "tl.LayerPassLog.func_config",
    "tl.LayerPassLog.func_is_inplace",
    "tl.LayerPassLog.func_kwargs_non_tensor",
    "tl.LayerPassLog.func_name",
    "tl.LayerPassLog.func_non_tensor_args",
    "tl.LayerPassLog.func_positional_args_non_tensor",
    "tl.LayerPassLog.func_rng_states",
    "tl.LayerPassLog.func_time",
    "tl.LayerPassLog.get_child_layers",
    "tl.LayerPassLog.get_parent_layers",
    "tl.LayerPassLog.grad_dtype",
    "tl.LayerPassLog.grad_fn_id",
    "tl.LayerPassLog.grad_fn_name",
    "tl.LayerPassLog.grad_fn_object",
    "tl.LayerPassLog.grad_memory",
    "tl.LayerPassLog.grad_memory_str",
    "tl.LayerPassLog.grad_shape",
    "tl.LayerPassLog.gradient",
    "tl.LayerPassLog.has_child_tensor_variations",
    "tl.LayerPassLog.has_children",
    "tl.LayerPassLog.has_co_parents",
    "tl.LayerPassLog.has_gradient",
    "tl.LayerPassLog.has_input_ancestor",
    "tl.LayerPassLog.has_internally_initialized_ancestor",
    "tl.LayerPassLog.has_parents",
    "tl.LayerPassLog.has_saved_activations",
    "tl.LayerPassLog.has_siblings",
    "tl.LayerPassLog.in_cond_branch",
    "tl.LayerPassLog.input_ancestors",
    "tl.LayerPassLog.internally_initialized_ancestors",
    "tl.LayerPassLog.internally_initialized_parents",
    "tl.LayerPassLog.intervention_log",
    "tl.LayerPassLog.io_role",
    "tl.LayerPassLog.is_buffer_layer",
    "tl.LayerPassLog.is_computed_inside_submodule",
    "tl.LayerPassLog.is_final_output",
    "tl.LayerPassLog.is_input_layer",
    "tl.LayerPassLog.is_internally_initialized",
    "tl.LayerPassLog.is_internally_terminated",
    "tl.LayerPassLog.is_leaf_module_output",
    "tl.LayerPassLog.is_output_ancestor",
    "tl.LayerPassLog.is_output_layer",
    "tl.LayerPassLog.is_part_of_iterable_output",
    "tl.LayerPassLog.is_scalar_bool",
    "tl.LayerPassLog.is_submodule_input",
    "tl.LayerPassLog.is_submodule_output",
    "tl.LayerPassLog.is_terminal_bool_layer",
    "tl.LayerPassLog.iterable_output_index",
    "tl.LayerPassLog.layer_label",
    "tl.LayerPassLog.layer_label_no_pass",
    "tl.LayerPassLog.layer_label_no_pass_short",
    "tl.LayerPassLog.layer_label_raw",
    "tl.LayerPassLog.layer_label_short",
    "tl.LayerPassLog.layer_label_w_pass",
    "tl.LayerPassLog.layer_label_w_pass_short",
    "tl.LayerPassLog.layer_total_num",
    "tl.LayerPassLog.layer_type",
    "tl.LayerPassLog.layer_type_num",
    "tl.LayerPassLog.leaf_module_pass",
    "tl.LayerPassLog.log_tensor_grad",
    "tl.LayerPassLog.lookup_keys",
    "tl.LayerPassLog.macs_backward",
    "tl.LayerPassLog.macs_forward",
    "tl.LayerPassLog.materialize_activation",
    "tl.LayerPassLog.materialize_gradient",
    "tl.LayerPassLog.max_distance_from_input",
    "tl.LayerPassLog.max_distance_from_output",
    "tl.LayerPassLog.min_distance_from_input",
    "tl.LayerPassLog.min_distance_from_output",
    "tl.LayerPassLog.module_address_normalized",
    "tl.LayerPassLog.module_entry_exit_thread_output",
    "tl.LayerPassLog.module_entry_exit_threads_inputs",
    "tl.LayerPassLog.module_nesting_depth",
    "tl.LayerPassLog.module_passes_entered",
    "tl.LayerPassLog.module_passes_exited",
    "tl.LayerPassLog.modules_entered",
    "tl.LayerPassLog.modules_entered_argnames",
    "tl.LayerPassLog.modules_exited",
    "tl.LayerPassLog.num_args",
    "tl.LayerPassLog.num_keyword_args",
    "tl.LayerPassLog.num_param_tensors",
    "tl.LayerPassLog.num_params_frozen",
    "tl.LayerPassLog.num_params_total",
    "tl.LayerPassLog.num_params_trainable",
    "tl.LayerPassLog.num_passes",
    "tl.LayerPassLog.num_positional_args",
    "tl.LayerPassLog.operation_equivalence_type",
    "tl.LayerPassLog.operation_num",
    "tl.LayerPassLog.output_descendants",
    "tl.LayerPassLog.output_device",
    "tl.LayerPassLog.output_path",
    "tl.LayerPassLog.params",
    "tl.LayerPassLog.params_memory",
    "tl.LayerPassLog.params_memory_str",
    "tl.LayerPassLog.parent_layer_arg_locs",
    "tl.LayerPassLog.parent_layers",
    "tl.LayerPassLog.parent_param_barcodes",
    "tl.LayerPassLog.parent_param_logs",
    "tl.LayerPassLog.parent_param_passes",
    "tl.LayerPassLog.parent_param_shapes",
    "tl.LayerPassLog.parent_params",
    "tl.LayerPassLog.pass_num",
    "tl.LayerPassLog.passes",
    "tl.LayerPassLog.recurrent_group",
    "tl.LayerPassLog.root_ancestors",
    "tl.LayerPassLog.save_gradients",
    "tl.LayerPassLog.save_tensor_data",
    "tl.LayerPassLog.scalar_bool_value",
    "tl.LayerPassLog.show",
    "tl.LayerPassLog.sibling_layers",
    "tl.LayerPassLog.source_model_log",
    "tl.LayerPassLog.tensor",
    "tl.LayerPassLog.tensor_dtype",
    "tl.LayerPassLog.tensor_label_raw",
    "tl.LayerPassLog.tensor_memory",
    "tl.LayerPassLog.tensor_memory_str",
    "tl.LayerPassLog.tensor_shape",
    "tl.LayerPassLog.transformed_activation",
    "tl.LayerPassLog.transformed_activation_dtype",
    "tl.LayerPassLog.transformed_activation_memory",
    "tl.LayerPassLog.transformed_activation_shape",
    "tl.LayerPassLog.transformed_gradient",
    "tl.LayerPassLog.transformed_gradient_dtype",
    "tl.LayerPassLog.transformed_gradient_memory",
    "tl.LayerPassLog.transformed_gradient_shape",
    "tl.LayerPassLog.uses_params",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")